# 04 · 主成分分析 Principal Component Analysis (PCA)

> **能力标签**：SH7（经典机器学习 / Classical ML）

## 目标 Objectives

完成本课后，你应该能够：

1. 理解**降维**（dimensionality reduction）的动机与 PCA 的线性代数基础。
2. 掌握**主成分**（principal components）与**解释方差比**（explained variance ratio）的含义。
3. 使用 `sklearn.decomposition.PCA` 拟合、变换并可视化主成分散点图。
4. 用碎石图（scree plot）确定合适的主成分数量。

> 对应能力：**SH7**

## 概念讲解 Concepts

### 为何降维？

高维数据面临：
- **维数灾难**（curse of dimensionality）：样本密度指数级稀疏。
- 可视化困难（人眼只能感知 ≤3 维）。
- 冗余特征（相关变量携带重复信息）。

---

### PCA 的核心思想

PCA 寻找数据方差最大的正交方向（**主成分**）作为新坐标轴：

1. **中心化**：$\tilde{X} = X - \bar{X}$
2. **协方差矩阵**：$C = \frac{1}{n-1}\tilde{X}^\top \tilde{X}$
3. **特征值分解**：$C = V \Lambda V^\top$（$V$ 的列 = 主成分方向）
4. **投影**：$Z = \tilde{X} V_k$（取前 $k$ 个特征向量）

**解释方差比**（explained variance ratio）：

$$\text{EVR}_j = \frac{\lambda_j}{\sum_i \lambda_i}$$

累计 EVR 通常用于选择 $k$：保留 95% 的总方差即可。

---

### 碎石图 Scree Plot

将 $\lambda_j$（或 EVR）按降序绘图，出现"拐点"（elbow）处即为合适的 $k$。

---

### 注意事项

- PCA 需要**先 z-score 标准化**（否则量纲大的特征主导）。
- 主成分是**无监督**的，不保证对分类/回归有判别力。

## 示例 Worked Example — `pca_transform` + 散点图

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tempfile
from pathlib import Path
from sklearn.datasets import make_classification
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ── pca_transform（镜像 w5-pca 题包）────────────────────────────────────────
def pca_transform(X, k: int):
    """PCA 降到 k 维，返回 (scores[n,k], explained_variance_ratio[k])。"""
    p = PCA(n_components=k)
    scores = p.fit_transform(np.asarray(X, dtype=float))
    return scores, p.explained_variance_ratio_

# ── 数据集（20 维 → 2 维可视化）──────────────────────────────────────────────
X, y = make_classification(n_samples=300, n_features=20, n_informative=6,
                            n_redundant=4, random_state=0)
scaler = StandardScaler()
X_s = scaler.fit_transform(X)

# 完整 PCA（保留所有成分，用于碎石图）
pca_full = PCA(n_components=None)
pca_full.fit(X_s)

# 2-D scores + scatter
scores_2d, evr_2d = pca_transform(X_s, k=2)
print(f"PC1 解释方差比 = {evr_2d[0]:.3f}  |  PC2 = {evr_2d[1]:.3f}")
print(f"前 2 主成分累计解释方差 = {evr_2d.sum():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# 散点图
for cls, color, label in [(0, "steelblue", "Class 0"), (1, "tomato", "Class 1")]:
    mask = (y == cls)
    axes[0].scatter(scores_2d[mask, 0], scores_2d[mask, 1],
                    c=color, alpha=0.6, s=20, label=label)
axes[0].set_xlabel(f"PC1 ({evr_2d[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({evr_2d[1]*100:.1f}%)")
axes[0].set_title("PCA 2-D 散点图（按类别着色）")
axes[0].legend()

# 碎石图（累计 EVR）
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
axes[1].bar(range(1, len(cumvar) + 1), pca_full.explained_variance_ratio_,
            color="steelblue", alpha=0.7, label="单个 PC")
axes[1].plot(range(1, len(cumvar) + 1), cumvar, "ro-", ms=4, label="累计 EVR")
axes[1].axhline(0.95, color="gray", ls="--", lw=1, label="95% 方差")
axes[1].set_xlabel("主成分编号")
axes[1].set_ylabel("解释方差比")
axes[1].set_title("碎石图（Scree Plot）")
axes[1].legend(fontsize=8)
axes[1].set_xlim(0.5, min(20, len(cumvar)) + 0.5)

fig.tight_layout()
_tmpdir = tempfile.mkdtemp()
_outpath = Path(_tmpdir) / "pca_demo.png"
fig.savefig(_outpath, dpi=80)
plt.close(fig)
print("图像已保存到:", _outpath)

## 动手 Exercise

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 练习：找到保留 90% / 95% 方差所需的最小主成分数
X, y = make_classification(n_samples=400, n_features=30, n_informative=10,
                            n_redundant=8, random_state=7)
scaler = StandardScaler()
X_s = scaler.fit_transform(X)

pca = PCA(n_components=None)
pca.fit(X_s)
cumvar = np.cumsum(pca.explained_variance_ratio_)

for threshold in [0.80, 0.90, 0.95, 0.99]:
    k = int(np.searchsorted(cumvar, threshold)) + 1
    print(f"保留 {threshold*100:.0f}% 方差需要 {k:>2} 个主成分（共 {X_s.shape[1]} 维）")

# 验证 pca_transform 输出形状
def pca_transform(X, k):
    p = PCA(n_components=k)
    scores = p.fit_transform(np.asarray(X, dtype=float))
    return scores, p.explained_variance_ratio_

scores, evr = pca_transform(X_s, k=5)
print(f"\npca_transform(X, k=5) -> scores.shape={scores.shape}, evr.shape={evr.shape}")

## 延伸阅读 Further Reading

1. **sklearn PCA**：<https://scikit-learn.org/stable/modules/decomposition.html#pca>
2. **3Blue1Brown — 线性代数可视化**：<https://www.youtube.com/watch?v=PFDu9oVAE-g>（特征值）
3. **《Pattern Recognition and Machine Learning》** Bishop — 第 12 章（连续潜变量模型）
4. **StatQuest — PCA Step by Step**：<https://www.youtube.com/watch?v=FgakZw6K1QQ>

## 关联练习 Related Assignments

```bash
python check.py w5-pca
```

> 实现 `pca_transform(X, k)` — PCA 降到 k 维，返回 `(scores[n,k], explained_variance_ratio[k])`。

> 能力标签：**SH7** · threshold ≥ 0.7